# EDA — NVDA MBP-1

Exploratory analysis of the order-update stream.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from utils import load_mbp1

df = load_mbp1("data/nvda_mbp1_2026-06-01.parquet")
df.shape

## Order updates per minute

Each row is one order-book update (add / cancel / trade). Resample the
event timestamps to 1-minute bins and count.

In [ ]:
# Count rows (order updates) in each 1-minute bin.
updates_per_min = (
    df.set_index("ts_event")
      .resample("1min")
      .size()
)

print(f"average updates per minute: {updates_per_min.mean()}")
updates_per_min

In [ ]:
# Plot it.
ax = updates_per_min.plot(figsize=(12, 4), title="NVDA order updates per minute (2026-06-01)")
ax.set_xlabel("Time (ET)")
ax.set_ylabel("# order updates")
plt.tight_layout()
plt.show()

## Factor visualization

Compute the five factors on the loaded book and inspect their behaviour
over the session and their distributions.

In [ ]:
from factors import (
    obi_ratio,
    net_liquidity_flow,
    trend_ratio,
    change_in_spread,
    volume_weighted_mid_deviation,
)

factors = pd.DataFrame({"ts_event": df["ts_event"]})
factors["obi_ratio"] = obi_ratio(df)
factors["net_liquidity_flow"] = net_liquidity_flow(df, look_back_ticks=100)
factors["trend_ratio"] = trend_ratio(df, look_back_ticks=100)
factors["change_in_spread"] = change_in_spread(df, look_back_ticks=100)
factors["vwmd"] = volume_weighted_mid_deviation(df, look_back_trades=100)
factors = factors.set_index("ts_event")

factor_cols = ["obi_ratio", "net_liquidity_flow", "trend_ratio", "change_in_spread", "vwmd"]
factors[factor_cols].describe()

### Time series (1-minute mean)

Each factor is computed per tick (~3.3M rows); we resample to the 1-minute
mean so the session-level behaviour is visible.

In [ ]:
per_min = factors[factor_cols].resample("1min").mean()

fig, axes = plt.subplots(len(factor_cols), 1, figsize=(12, 12), sharex=True)
for ax, col in zip(axes, factor_cols):
    per_min[col].plot(ax=ax)
    ax.set_ylabel(col)
    ax.axhline(0, color="grey", lw=0.6, ls="--")
axes[0].set_title("NVDA factors — 1-minute mean (2026-06-01)")
axes[-1].set_xlabel("Time (ET)")
plt.tight_layout()
plt.show()

### Distributions

Per-tick distribution of each factor (NaN warm-up rows dropped automatically).

In [ ]:
fig, axes = plt.subplots(1, len(factor_cols), figsize=(18, 3))
for ax, col in zip(axes, factor_cols):
    factors[col].plot.hist(bins=100, ax=ax)
    ax.set_title(col)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()